# Aerodynamics

The attitude-sensitive component model (`lapsim/subsystems/aero.py`): front wing, rear wing and body, each
with its own downforce, drag, centre of pressure, pitch and yaw sensitivity and (front wing) ground effect,
scaled to the measured whole-car ClA / CdA at the coast-down speed. The vehicle model finds the ride
height, pitch, roll and body slip the aero loads produce and feeds them back (`core/vehicle.py`).

Move a slider: downforce, drag, aero balance and the component table update at once. **Run laps** shows
what the change does to the event times.

In [ ]:
%matplotlib inline
from nb import *

In [ ]:
car_pick = W.ToggleButtons(options=list(CARS), description="Car")
out_fast, out_lap = W.Output(), W.Output()
box = W.VBox()


def build(_=None):
    global CAR, SL
    CAR = lapsim.load_car(CARS[car_pick.value])
    SL = [slider(CAR, "vehicle:aero.calib_ClA", 0.0, 4.0, 0.05, "whole-car ClA", 1, "m^2"),
          slider(CAR, "vehicle:aero.calib_CdA", 0.4, 2.0, 0.02, "whole-car CdA", 1, "m^2"),
          slider(CAR, "vehicle:aero.calib_front_axle_share", 0.30, 0.70, 0.005, "front-axle share of aero load"),
          slider(CAR, "vehicle:mass.cg_height", 200, 350, 1, "CG height", 1e3, "mm")]
    for s in SL:
        s.observe(lambda ch: draw(), "value")
    box.children = [*SL, run_button("Run laps", laps, out_lap)]
    draw()


def draw():
    with out_fast:
        out_fast.clear_output(wait=True)
        veh = with_values(CAR, SL).build()
        v = np.linspace(5, 35, 31)
        f, r, d = veh.aero(v)
        fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
        ax[0].plot(3.6 * v, f + r, label="downforce"); ax[0].plot(3.6 * v, d, label="drag")
        ax[0].set(xlabel="speed [km/h]", ylabel="N", title="Straight line, aero-loaded ride height"); ax[0].legend()
        ax[1].plot(3.6 * v, 100 * f / (f + r)); ax[1].set(xlabel="speed [km/h]", ylabel="%", title="Front-axle share")
        ay = np.linspace(0, 1.6, 9) * 9.81
        st = veh.state(np.full_like(ay, 15.0), 0 * ay, ay)
        ax[2].plot(ay / 9.81, st["downforce"], label="downforce at 15 m/s")
        ax[2].plot(ay / 9.81, st["drag"], label="drag")
        ax[2].set(xlabel="lateral acceleration [g]", ylabel="N", title="Cornering: roll, yaw and slip"); ax[2].legend()
        plt.show()
        display(veh.aero_model.table().round(3))


def laps():
    res = with_values(CAR, SL).run(("acceleration", "skidpad", "autocross"))
    display(res.table().round(3))


car_pick.observe(build, "value")
build()
display(W.VBox([car_pick, box, out_fast, out_lap]))